In [1]:
# if the following command generates an error, you probably didn't enable 
# the cluster security option "Allow API access to all Google Cloud services"
# under Manage Security → Project Access when setting up the cluster
!gcloud dataproc clusters list --region us-central1

NAME          PLATFORM  PRIMARY_WORKER_COUNT  SECONDARY_WORKER_COUNT  STATUS   ZONE           SCHEDULED_DELETE  SCHEDULED_STOP
cluster-0016  GCE       2                                             RUNNING  us-central1-a


# Imports & Setup

In [ ]:
# !pip install -q google-cloud-storage==1.43.0
!pip install google-cloud-storage>=2.10.0
!pip install -q graphframes

In [ ]:
import pyspark
import sys
from collections import Counter, OrderedDict, defaultdict
import itertools
from itertools import islice, count, groupby
import pandas as pd
import os
import re
from operator import itemgetter
import nltk
from nltk.stem.porter import *
from nltk.corpus import stopwords
from pathlib import Path
import pickle
import pandas as pd
from google.cloud import storage

import hashlib
def _hash(s):
    return hashlib.blake2b(bytes(s, encoding='utf8'), digest_size=5).hexdigest()

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [9]:
# if nothing prints here you forgot to include the initialization script when starting the cluster
!ls -l /usr/lib/spark/jars/graph*

-rw-r--r-- 1 root root 247882 Dec  7 17:24 /usr/lib/spark/jars/graphframes-0.8.2-spark3.1-s_2.12.jar


In [10]:
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark import SparkContext, SparkConf, SparkFiles
from pyspark.sql import SQLContext
from graphframes import *

In [11]:
spark

In [15]:
# Put your bucket name below and make sure you can access it without an error
bucket_name = 'ir_bgu_322793365' 
full_path = f"gs://{bucket_name}/"
paths=[]

client = storage.Client()
blobs = client.list_blobs(bucket_name)
for b in blobs:
    if b.name != 'graphframes.sh':
        paths.append(full_path+b.name)

***GCP setup is complete!*** 

# Building an inverted index

Here, we read the entire corpus to an rdd, directly from Google Storage Bucket and use your code from Colab to construct an inverted index.

In [ ]:
parquetFile = spark.read.parquet(f"gs://{bucket_name}/multistream*")

doc_text_pairs  = parquetFile.select("text", "id").rdd
doc_title_pairs = parquetFile.select("title", "id").rdd
doc_anchor_text_pairs = parquetFile.select("anchor_text", "id").rdd

flattened_doc_anchor_pairs = doc_anchor_text_pairs.flatMap(
    lambda rec: [(a.text, a.id) for a in rec[0] if getattr(a, "text", None)]
)

We will count the number of pages to make sure we are looking at the entire corpus. The number of pages should be more than 6M

In [17]:
# Count number of wiki pages
parquetFile.count()

6348910

Let's import the inverted index module. Note that you need to use the staff-provided version called `inverted_index_gcp.py`, which contains helper functions to writing and reading the posting files similar to the Colab version, but with writing done to a Google Cloud Storage bucket.

In [18]:
# if nothing prints here you forgot to upload the file inverted_index_gcp.py to the home dir
%cd -q /home/dataproc
!ls inverted_index_gcp.py

inverted_index_gcp.py


In [21]:
# adding our python module to the cluster
sc.addFile("/home/dataproc/inverted_index_gcp.py")
sys.path.insert(0,SparkFiles.getRootDirectory())

25/12/07 17:37:53 WARN org.apache.spark.SparkContext: The path /home/dataproc/inverted_index_gcp.py has been added already. Overwriting of added paths is not supported in the current version.


In [22]:
from inverted_index_gcp import InvertedIndex

In [ ]:
english_stopwords = frozenset(stopwords.words('english'))
corpus_stopwords = ["category", "references", "also", "external", "links", 
                    "may", "first", "see", "history", "people", "one", "two", 
                    "part", "thumb", "including", "second", "following", 
                    "many", "however", "would", "became"]

all_stopwords = english_stopwords.union(corpus_stopwords)
RE_WORD = re.compile(r"""[\#\@\w](['\-]?\w){2,24}""", re.UNICODE)

NUM_BUCKETS = 124
def token2bucket_id(token):
  return int(_hash(token),16) % NUM_BUCKETS

# PLACE YOUR CODE HERE
def word_count(text, id):
  ''' Count the frequency of each word in `text` (tf) that is not included in
  `all_stopwords` and return entries that will go into our posting lists.
  Parameters:
  -----------
    text: str
      Text of one document
    id: int
      Document id
  Returns:
  --------
    List of tuples
      A list of (token, (doc_id, tf)) pairs
      for example: [("Anarchism", (12, 5)), ...]
  '''
  tokens = [token.group() for token in RE_WORD.finditer(text.lower())]
  tf_counter = Counter(token for token in tokens if token not in all_stopwords)
  return [(token, (id, tf)) for token, tf in tf_counter.items()]

def reduce_word_counts(unsorted_pl):
  ''' Returns a sorted posting list by wiki_id.
  Parameters:
  -----------
    unsorted_pl: list of tuples
      A list of (wiki_id, tf) tuples
  Returns:
  --------
    list of tuples
      A sorted posting list.
  '''
  return sorted(unsorted_pl, key=lambda x: x[0])

def calculate_df(postings):
  ''' Takes a posting list RDD and calculate the df for each token.
  Parameters:
  -----------
    postings: RDD
      An RDD where each element is a (token, posting_list) pair.
  Returns:
  --------
    RDD
      An RDD where each element is a (token, df) pair.
  '''
  return postings.map(lambda x: (x[0], len(x[1])))

def partition_postings_and_write(postings, base_dir='postings_gcp'):
  ''' A function that partitions the posting lists into buckets, writes out
  all posting lists in a bucket to disk, and returns the posting locations for
  each bucket. Partitioning should be done through the use of `token2bucket`
  above. Writing to disk should use the function  `write_a_posting_list`, a
  static method implemented in inverted_index_colab.py under the InvertedIndex
  class.
  Parameters:
  -----------
    postings: RDD
      An RDD where each item is a (w, posting_list) pair.
  Returns:
  --------
    RDD
      An RDD where each item is a posting locations dictionary for a bucket. The
      posting locations maintain a list for each word of file locations and
      offsets its posting list was written to. See `write_a_posting_list` for
      more details.
  '''
  return (postings
          .map(lambda x: (token2bucket_id(x[0]), x))
          .groupByKey()
          .map(lambda x: InvertedIndex.write_a_posting_list(
              (x[0], list(x[1])), 
              base_dir,        # This is the directory prefix
              'ir-project-322793365'         # This is the bucket name variable from the global scope
          )))

def construct_inverted_index(pairs, index_prefix='postings_gcp', apply_filter=True):
    word_counts = pairs.flatMap(lambda x: word_count(x[0], x[1]))
    postings = word_counts.groupByKey().mapValues(reduce_word_counts)
    # filtering postings and calculate df - relevant for text only
    if (apply_filter):
        postings = postings.filter(lambda x: len(x[1])>50)
    w2df = calculate_df(postings)
    w2df_dict = w2df.collectAsMap()
    # partition posting lists and write out
    _ = partition_postings_and_write(postings, index_prefix).collect()

    super_posting_locs = defaultdict(list)
    for blob in client.list_blobs(bucket_name, prefix=index_prefix):
        if not blob.name.endswith("pickle"):
            continue
        with blob.open("rb") as f:
            posting_locs = pickle.load(f)
            for k, v in posting_locs.items():
                super_posting_locs[k].extend(v)
    
    # Create inverted index instance
    inverted = InvertedIndex()
    # Adding the posting locations dictionary to the inverted index
    inverted.posting_locs = super_posting_locs
    # Add the token - df dictionary to the inverted index
    inverted.df = w2df_dict
    return inverted

def write_inverted_index(inverted_index, index_name, index_prefix='postings_gcp'):
    inverted_index.write_index('.', index_name)
    # upload to gs
    index_src = f"{index_name}.pkl"
    index_dst = f'gs://{bucket_name}/{index_prefix}/{index_src}'
    !gsutil cp $index_src $index_dst

In [ ]:
# word counts map
word_counts = doc_title_pairs.flatMap(lambda x: word_count(x[0], x[1]))
postings = word_counts.groupByKey().mapValues(reduce_word_counts)
w2df = calculate_df(postings)
w2df_dict = w2df.collectAsMap()
# partition posting lists and write out
_ = partition_postings_and_write(postings).collect()

In [ ]:
super_posting_locs = defaultdict(list)
for blob in client.list_blobs(bucket_name, prefix='postings_gcp'):
  if not blob.name.endswith("pickle"):
    continue
  with blob.open("rb") as f:
    posting_locs = pickle.load(f)
    for k, v in posting_locs.items():
      super_posting_locs[k].extend(v)

In [ ]:
# Create inverted index instance
inverted = InvertedIndex()
# Adding the posting locations dictionary to the inverted index
inverted.posting_locs = super_posting_locs
# Add the token - df dictionary to the inverted index
inverted.df = w2df_dict
# write the global stats out
inverted.write_index('.', 'title_index')
# upload to gs
index_src = "title_index.pkl"
index_dst = f'gs://{bucket_name}/postings_gcp/{index_src}'
!gsutil cp $index_src $index_dst

In [ ]:
!gsutil ls -lh $index_dst

In [ ]:
# word counts map
word_counts = flattened_doc_anchor_pairs.flatMap(lambda x: word_count(x[0], x[1]))
postings = word_counts.groupByKey().mapValues(reduce_word_counts)
w2df = calculate_df(postings)
w2df_dict = w2df.collectAsMap()
# partition posting lists and write out
_ = partition_postings_and_write(postings).collect()

In [ ]:
super_posting_locs = defaultdict(list)
for blob in client.list_blobs(bucket_name, prefix='postings_gcp'):
  if not blob.name.endswith("pickle"):
    continue
  with blob.open("rb") as f:
    posting_locs = pickle.load(f)
    for k, v in posting_locs.items():
      super_posting_locs[k].extend(v)

In [ ]:
# Create inverted index instance
inverted = InvertedIndex()
# Adding the posting locations dictionary to the inverted index
inverted.posting_locs = super_posting_locs
# Add the token - df dictionary to the inverted index
inverted.df = w2df_dict
# write the global stats out
inverted.write_index('.', 'anchor_index')
# upload to gs
index_src = "anchor_index.pkl"
index_dst = f'gs://{bucket_name}/postings_gcp/{index_src}'
!gsutil cp $index_src $index_dst

In [ ]:
!gsutil ls -lh $index_dst

In [ ]:
text_index = construct_inverted_index(
    doc_text_pairs, 
    index_prefix='postings_gcp/text',
    apply_filter=True
)
write_inverted_index(text_index, 'text_index', index_prefix='postings_gcp/text')


In [ ]:
title_index = construct_inverted_index(
    doc_title_pairs, 
    index_prefix='postings_gcp/title',
    apply_filter=False
)
write_inverted_index(title_index, 'title_index', index_prefix='postings_gcp/title')


In [ ]:
# Build anchor index
anchor_index = construct_inverted_index(
    flattened_doc_anchor_pairs,
    index_prefix='postings_gcp/anchor',
    apply_filter=False
)
write_inverted_index(anchor_index, 'anchor_index', index_prefix='postings_gcp/anchor')


# PageRank

In [43]:
# Put your `generate_graph` function here
def generate_graph(pages):
  ''' Compute the directed graph generated by wiki links.
  Parameters:
  -----------
    pages: RDD
      An RDD where each row consists of one wikipedia articles with 'id' and
      'anchor_text'.
  Returns:
  --------
    edges: RDD
      An RDD where each row represents an edge in the directed graph created by
      the wikipedia links. The first entry should the source page id and the
      second entry is the destination page id. No duplicates should be present.
    vertices: RDD
      An RDD where each row represents a vetrix (node) in the directed graph
      created by the wikipedia links. No duplicates should be present.
  '''
# Extract edges from (source_id, dest_id) for valid links
  edges = pages.flatMap(
        lambda x: [(x['id'], dest['id']) for dest in x['anchor_text'] if 'id' in dest and dest['id'] is not None]
    ).distinct()

# Extract vertices directly from edges
  vertices = edges.flatMap(lambda x: x).distinct().map(lambda v: (v,))

  return edges, vertices

In [ ]:
pages_links = spark.read.parquet("gs://ir_bgu_322793365/multistream*").select("id", "anchor_text").rdd
# construct the graph 
edges, vertices = generate_graph(pages_links)
# compute PageRank
edgesDF = edges.toDF(['src', 'dst']).repartition(124, 'src')
verticesDF = vertices.toDF(['id']).repartition(124, 'id')
g = GraphFrame(verticesDF, edgesDF)
pr_results = g.pageRank(resetProbability=0.15, maxIter=6)
pr = pr_results.vertices.select("id", "pagerank")
pr = pr.sort(col('pagerank').desc())
pr.repartition(1).write.csv(f'gs://{bucket_name}/pr', compression="gzip")
pr.show()

+-------+------------------+
|     id|          pagerank|
+-------+------------------+
|3434750| 9913.728782160773|
|  10568| 5385.349263642038|
|  32927| 5282.081575765277|
|  30680| 5128.233709604119|
|5843419| 4957.567686263868|
|  68253|  4769.27826535516|
|  31717|  4486.35018054831|
|  11867|4146.4146509127695|
|  14533|3996.4664408855037|
| 645042|3531.6270898037424|
|  17867|3246.0983906041415|
|5042916| 2991.945739166177|
|4689264| 2982.324883041747|
|  14532| 2934.746829203171|
|  25391| 2903.546223513398|
|   5405| 2891.416329154636|
|4764461| 2834.366987332661|
|  15573| 2783.865118158839|
|   9316|2782.0396464137693|
|8569916| 2775.286191840016|
+-------+------------------+
only showing top 20 rows



# Page views

In [ ]:
import pickle
from collections import Counter

pv_path = 'https://dumps.wikimedia.org/other/pageview_complete/monthly/2021/2021-08/pageviews-202108-user.bz2'
p = Path(pv_path) 
pv_name = p.name
pv_temp = f'{p.stem}-4dedup.txt'
pv_clean = f'{p.stem}.pkl'
# # Download the file (2.3GB) 
!wget -N $pv_path
# # Filter for English pages, and keep just two fields: article ID (3) and monthly 
# # total number of page views (5). Then, remove lines with article id or page 
# # view values that are not a sequence of digits.
!bzcat $pv_name | grep "^en\.wikipedia" | cut -d' ' -f3,5 | grep -P "^\d+\s\d+$" > $pv_temp
# # Create a Counter (dictionary) that sums up the pages views for the same 
# # article, resulting in a mapping from article id to total page views.
wid2pv = Counter()
with open(pv_temp, 'rt') as f:
  for line in f:
    parts = line.split(' ')
    wid2pv.update({int(parts[0]): int(parts[1])})
# # write out the counter as binary file (pickle it)
# with open(pv_clean, 'wb') as f:
#   pickle.dump(wid2pv, f)
# with open(pv_clean, 'rb') as f:
#   wid2pv = pickle.loads(f.read())
page_view_dict = defaultdict(int)
for doc_id, view in wid2pv.items():
  page_view_dict[doc_id] = view

with open("pageview.pkl", 'wb') as f:
  pickle.dump(page_view_dict, f)

# bucket = client.bucket(bucket_name)
# blob = bucket.blob('pv/pageview.pkl')
# blob.upload_from_filename('pageview.pkl')

# Title Mappings

In [ ]:
# Create document ID to Title mapping
print("Creating document title mappings...")

# Read titles from parquet
pages_data = spark.read.parquet(f"gs://{bucket_name}/multistream*").select("id", "title")

# Collect as dictionary
id_title_dict = pages_data.rdd.map(lambda row: (int(row['id']), row['title'])).collectAsMap()

# Split into even/odd for smaller file sizes (memory efficiency)
even_titles = {doc_id: title for doc_id, title in id_title_dict.items() if doc_id % 2 == 0}
odd_titles = {doc_id: title for doc_id, title in id_title_dict.items() if doc_id % 2 == 1}

print(f"Even titles: {len(even_titles)}, Odd titles: {len(odd_titles)}")

# Save locally
with open('even_id_title_dict.pkl', 'wb') as f:
    pickle.dump(even_titles, f)

with open('uneven_id_title_dict.pkl', 'wb') as f:
    pickle.dump(odd_titles, f)

# Upload to GCS
bucket = client.bucket(bucket_name)

even_blob = bucket.blob('id_title/even_id_title_dict.pkl')
even_blob.upload_from_filename('even_id_title_dict.pkl')

odd_blob = bucket.blob('id_title/uneven_id_title_dict.pkl')
odd_blob.upload_from_filename('uneven_id_title_dict.pkl')

print("✅ Title mappings uploaded to GCS")

# embedding indecis

In [ ]:
# Document Embeddings Creation
print("Creating document embeddings...")

# Download word2vec model (or use any pre-trained model)
import gensim.downloader as api
print("Loading word2vec model...")
w2v_model = api.load('word2vec-google-news-300')  # 300-dim vectors

def text_to_embedding(text, model):
    """
    Convert text to embedding by averaging word vectors.
    Parameters:
    -----------
        text: str - document text
        model: word2vec model
    Returns:
    --------
        numpy array of shape (300,) or None if no valid words
    """
    import numpy as np
    tokens = [token.group().lower() for token in RE_WORD.finditer(text) 
              if token.group().lower() not in all_stopwords]
    
    valid_vectors = []
    for token in tokens:
        if token in model:
            valid_vectors.append(model[token])
    
    if len(valid_vectors) == 0:
        return np.zeros(300)  # Return zero vector if no valid words
    
    return np.mean(valid_vectors, axis=0)

# Process title embeddings (faster, recommended for reranking)
print("Processing title embeddings...")
title_embeddings_rdd = doc_title_pairs.map(
    lambda x: (int(x[1]), text_to_embedding(x[0], w2v_model))
)

# Collect embeddings as list of (doc_id, vector) tuples
title_embeddings = title_embeddings_rdd.collect()

# Convert to numpy arrays for efficient storage
import numpy as np
doc_ids = np.array([doc_id for doc_id, _ in title_embeddings], dtype=np.int32)
embeddings = np.array([vec for _, vec in title_embeddings], dtype=np.float32)

print(f"Created embeddings for {len(doc_ids)} documents")
print(f"Embedding shape: {embeddings.shape}")

# Save locally
np.save('title_doc_ids.npy', doc_ids)
np.save('title_embeddings.npy', embeddings)

# Upload to GCS
embeddings_path = 'embeddings/title'
!gsutil cp title_doc_ids.npy gs://{bucket_name}/{embeddings_path}/title_doc_ids.npy
!gsutil cp title_embeddings.npy gs://{bucket_name}/{embeddings_path}/title_embeddings.npy

print("✅ Title embeddings uploaded to GCS")

# Optional: Create body text embeddings (slower, more memory intensive)
# Uncomment if needed:
# print("Processing body text embeddings...")
# body_embeddings_rdd = doc_text_pairs.map(
#     lambda x: (int(x[1]), text_to_embedding(x[0][:5000], w2v_model))  # Limit to first 5000 chars
# )
# body_embeddings = body_embeddings_rdd.collect()
# body_doc_ids = np.array([doc_id for doc_id, _ in body_embeddings], dtype=np.int32)
# body_vecs = np.array([vec for _, vec in body_embeddings], dtype=np.float32)
# np.save('body_doc_ids.npy', body_doc_ids)
# np.save('body_embeddings.npy', body_vecs)
# !gsutil cp body_doc_ids.npy gs://{bucket_name}/embeddings/body/body_doc_ids.npy
# !gsutil cp body_embeddings.npy gs://{bucket_name}/embeddings/body/body_embeddings.npy

In [46]:
# size of input data
!gsutil du -sh "gs://wikidata_preprocessed/"

14.28 GiB    gs://wikidata_preprocessed


In [ ]:
# size of index data
index_dst = f'gs://{bucket_name}/postings_gcp/'
!gsutil du -sh "$index_dst"

5.92 GiB     gs://ir_bgu_322793365/postings_gcp
